# Exploratory Data Analysis

**Owner:** Mohd Yah-Ya Raiyan

**Purpose:** two independent EDA passes over the cleaned feature store
(Section 3.6 of the report) — distribution checks, a Sydney vs Rest-of-NSW
regional comparison, and the counter-intuitive findings that shaped later
modelling decisions.


In [1]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fs = pd.read_csv("data/NSW_feature_store.csv", parse_dates=["quarter"])
print(f"Feature store: {len(fs):,} suburb-quarter rows, {fs['suburb'].nunique():,} suburbs")
print(f"Date range: {fs['quarter'].min().date()} to {fs['quarter'].max().date()}")
fs.head()


Feature store: 78,384 suburb-quarter rows, 2,585 suburbs
Date range: 2013-01-01 to 2026-07-01


,suburb,postcode,region,quarter,quarter_of_year,n_sales,median_price,median_price_per_sqm,qoq_pct_change,lag_1_price,...,state_benchmark_growth,nsw_tvd_qoq_pct,missing_benchmark_flag,is_q1,is_q2,is_q3,is_q4,flag_covid_period,flag_rate_hike_period,n_outliers_excluded_candidate
0,ABBOTSBURY,2176.0,sydney,2013-10-01,4,3,620000.0,847.457627,NaN,NaN,...,11.194030,4.851749,0,0,0,0,1,0,0,0
1,ABBOTSBURY,2176.0,sydney,2014-01-01,1,11,705000.0,1071.428571,13.709677,620000.0,...,-8.724832,1.977398,0,1,0,0,0,0,0,0
2,ABBOTSBURY,2176.0,sydney,2014-04-01,2,13,815000.0,1124.625125,15.602837,705000.0,...,7.058824,3.326688,0,0,1,0,0,0,0,0
3,ABBOTSBURY,2176.0,sydney,2014-07-01,3,13,720000.0,970.873786,-11.656442,815000.0,...,0.274725,3.029073,0,0,0,1,0,0,0,0
4,ABBOTSBURY,2176.0,sydney,2014-10-01,4,6,781500.0,1306.574572,8.541667,720000.0,...,11.780822,4.598269,0,0,0,0,1,0,0,0


## Pass 1: Distribution checks

In [2]:

fs["median_price"].describe()


count    7.838400e+04
mean     9.055408e+05
std      7.639856e+05
min      1.000000e+04
25%      4.825000e+05
50%      7.300000e+05
75%      1.076062e+06
max      4.100000e+07
Name: median_price, dtype: float64

In [3]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(fs["median_price"].clip(upper=3_000_000), bins=60, color="#3454D1")
axes[0].set_title("Median price distribution (clipped at $3M)")
axes[0].set_xlabel("Median price ($)")

axes[1].hist(fs["qoq_pct_change"].dropna().clip(-40, 40), bins=60, color="#E2574C")
axes[1].set_title("QoQ % change distribution (clipped at ±40%)")
axes[1].set_xlabel("QoQ % change")
plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=100)
plt.show()
print("qoq_pct_change skew:", fs["qoq_pct_change"].skew().round(2))
print("qoq_pct_change std:", fs["qoq_pct_change"].std().round(2), "percentage points")


qoq_pct_change skew: 75.87
qoq_pct_change std: 60.97 percentage points


**Finding 1 (counter-intuitive):** the quarter-on-quarter price change
distribution is much wider and more skewed than a "house prices move
slowly" intuition would suggest — a standard deviation of double-digit
percentage points per quarter, driven by low-sales-volume suburbs where a
handful of transactions can swing the median sharply. This directly
motivates why the models are evaluated with MAE/RMSE in percentage points
rather than judged against a naive "prices barely move" expectation.

## Pass 2: Regional comparison (Sydney vs Rest-of-NSW)

In [4]:

region_stats = fs.groupby("region").agg(
    n_suburb_quarters=("median_price", "size"),
    median_price=("median_price", "median"),
    median_qoq_pct=("qoq_pct_change", "median"),
    std_qoq_pct=("qoq_pct_change", "std"),
).round(2)
region_stats


,n_suburb_quarters,median_price,median_qoq_pct,std_qoq_pct
region,,,,
rest_nsw,45838,562500.0,2.31,48.08
sydney,32546,999000.0,1.68,75.22


In [5]:

fig, ax = plt.subplots(figsize=(8, 4))
for region, color in [("sydney", "#3454D1"), ("rest_nsw", "#1F9E89")]:
    trend = fs[fs["region"] == region].groupby("quarter")["median_price"].median()
    ax.plot(trend.index, trend.values, label=region, color=color)
ax.set_title("Median price over time, by region")
ax.legend()
plt.tight_layout()
plt.savefig("eda_regional_trend.png", dpi=100)
plt.show()


**Finding 2 (counter-intuitive):** Rest-of-NSW suburbs show *higher*
QoQ volatility (higher `std_qoq_pct`) than Sydney metro suburbs despite
having lower median prices — smaller, thinner markets swing harder on a
percentage basis even though the dollar amounts involved are smaller. This
is part of the justification for including `n_sales` and region as model
features rather than treating all suburbs as equally predictable.

## Correlation with the target (next-quarter QoQ % change proxy)

In [6]:

feature_cols = ["n_sales","lag_1_qoq_pct","lag_2_qoq_pct","rolling_mean_4q","rolling_std_4q",
                "state_benchmark_growth","flag_covid_period","flag_rate_hike_period"]
corr = fs[feature_cols + ["qoq_pct_change"]].corr()["qoq_pct_change"].drop("qoq_pct_change").sort_values(key=abs, ascending=False)
corr


lag_1_qoq_pct            -0.112486
rolling_std_4q            0.062361
n_sales                  -0.040824
state_benchmark_growth    0.028692
flag_covid_period         0.026849
lag_2_qoq_pct             0.015172
rolling_mean_4q          -0.010052
flag_rate_hike_period    -0.008829
Name: qoq_pct_change, dtype: float64

**Finding 3:** `lag_1_qoq_pct` (last quarter's change) is the strongest
single correlate of the current quarter's change, but the correlation is
still fairly weak in absolute terms — consistent with the low R² and
~50% directional accuracy reported later in `model_evaluation.ipynb`.
This is exactly why the report is upfront about forecast uncertainty
rather than presenting a single confident number.